# 🐴 期待値算出・購入候補出力

今週の出走馬の期待値を算出して購入候補を出力します。

**使い方：**
1. セル①〜②を実行
2. セル③で対象レースの日付を指定
3. セル④〜⑤で期待値上位馬を確認

In [ ]:
!pip install supabase lightgbm scikit-learn -q

In [ ]:
SUPABASE_URL = 'https://infypumigexmpdmijhnx.supabase.co'
SUPABASE_KEY = 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9...'  # service_role key
from supabase import create_client
import pandas as pd, joblib
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
model = joblib.load('keiba_model_XXXXXXXX.pkl')  # ファイル名を変更
print('✅ 準備完了')

In [ ]:
# 対象日付を指定（今週の開催日）
TARGET_DATE_FROM = '2026-05-23'  # ← ここを変更
TARGET_DATE_TO   = '2026-05-25'  # ← ここを変更

all_data = []
offset = 0
while True:
    res = supabase.table('v_features').select('*') \
        .gte('race_date', TARGET_DATE_FROM) \
        .lte('race_date', TARGET_DATE_TO) \
        .range(offset, offset+999).execute()
    if not res.data: break
    all_data.extend(res.data)
    offset += 1000
df = pd.DataFrame(all_data)
print(f'✅ {len(df):,}件取得 ({TARGET_DATE_FROM} 〜 {TARGET_DATE_TO})')

In [ ]:
from sklearn.preprocessing import LabelEncoder
cat_cols = ['venue_code', 'surface', 'track_condition', 'weather', 'class']
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
feature_cols = [
    'venue_code_enc', 'distance', 'surface_enc', 'track_condition_enc',
    'weather_enc', 'post_position', 'entry_count', 'class_enc',
    'avg_finish_3', 'avg_finish_5', 'fukusho_rate_3', 'fukusho_rate_5',
    'last_finish', 'rest_days', 'distance_diff', 'horse_weight', 'horse_weight_diff',
    'jockey_fukusho_rate', 'trainer_fukusho_rate',
    'father_fukusho_rate', 'mother_father_fukusho_rate',
    'odds', 'popularity', 'popularity_ratio',
]
df['predicted_proba'] = model.predict_proba(df[feature_cols])[:, 1]

In [ ]:
res = supabase.table('payouts').select('race_id,combination,payout').eq('bet_type', '複勝').execute()
df_pay = pd.DataFrame(res.data) if res.data else pd.DataFrame(columns=['race_id','combination','payout'])
if len(df_pay) > 0:
    df_pay['horse_number'] = df_pay['combination'].str.extract(r'(\d+)').astype(float)
    df_pay['fukusho_odds'] = df_pay['payout'] / 100
    df = df.merge(df_pay[['race_id','horse_number','fukusho_odds']], on=['race_id','horse_number'], how='left')
else:
    df['fukusho_odds'] = df['odds'] * 0.4  # 未確定時は単勝から推定
df['expected_value'] = df['predicted_proba'] * df['fukusho_odds'].fillna(0)
print('✅ 期待値算出完了')

In [ ]:
# 購入候補出力（期待値 > 1.0）
buy = df[df['expected_value'] > 1.0].copy()
buy = buy.sort_values('expected_value', ascending=False)
horses = pd.DataFrame(supabase.table('horses').select('horse_id,horse_name').execute().data)
buy = buy.merge(horses, on='horse_id', how='left')
display_cols = ['race_date','venue_code','race_id','horse_number','horse_name',
                'popularity','odds','predicted_proba','fukusho_odds','expected_value']
print(f'\n🎯 購入候補: {len(buy)}件（期待値 > 1.0）')
print('=' * 80)
buy[display_cols]